# Caso 2 — Modelo Vetorial (Vector Space Model)

Este notebook implementa o **Modelo Vetorial** clássico: cada documento e
cada consulta são representados como vetores no espaço de termos, com peso
TF-IDF, e o ranking é dado pela similaridade de cosseno entre o vetor da
consulta e os vetores dos documentos.

A implementação (em `src/vector_model.py`) calcula manualmente tf, idf e o
cosseno — nenhuma parte do cálculo do score é delegada a uma função pronta
de biblioteca (ex.: `TfidfVectorizer`/`cosine_similarity` do scikit-learn).
`scipy.sparse` é usado apenas como estrutura de dados para vetores esparsos.


In [1]:
from pathlib import Path
import sys
project_root = Path.cwd()
if not (project_root / "src").is_dir() and (project_root.parent / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from src.cranfield_data import load_cranfield
from src.pre_processing import load_preprocessed
from src.vector_model import VectorSpaceModel

df_docs, df_queries, df_qrels = load_cranfield()
preprocessed = load_preprocessed(project_root / "data" / "processed" / "preprocessed_cranfield.pkl")

doc_ids = df_docs["doc_id"].tolist()
query_ids = df_queries["query_id"].tolist()
doc_title = dict(zip(df_docs.doc_id, df_docs.title.str.replace("\n", " ")))


[cranfield_data] ir_datasets indisponível (RuntimeError: ('All download sources failed', [(RequestsDownload('http://ir.dcs.gla.ac.uk/resources/test_collections/cran/cran.tar.gz', tries=None), HTTPError('403 Client Error: Forbidden for url: http://ir.dcs.gla.ac.uk/resources/test_collections/cran/cran.tar.gz')), (RequestsDownload('https://mirror.ir-datasets.com/1730f7be572d95a5a4b56c59a7b900a5', tries=None), HTTPError('403 Client Error: Forbidden for url: https://mirror.ir-datasets.com/1730f7be572d95a5a4b56c59a7b900a5'))])). Usando arquivos locais em data/raw/ como fallback.


[WARNING] Download failed: 403 Client Error: Forbidden for url: http://ir.dcs.gla.ac.uk/resources/test_collections/cran/cran.tar.gz
[WARNING] Download failed: 403 Client Error: Forbidden for url: https://mirror.ir-datasets.com/1730f7be572d95a5a4b56c59a7b900a5


## Configuração de pré-processamento adotada

A partir deste ponto, fixamos uma única configuração de pré-processamento
para permitir comparações justas entre os modelos nos casos seguintes.


In [2]:
# Configuração de pré-processamento usada a partir daqui
#
# O Caso 1 comparou 4 configurações de pré-processamento (sem stopwords/sem
# stemming; com stopwords; com stemming; com stopwords+stemming). A partir
# deste notebook, todos os modelos de recuperação (Modelo Vetorial e BM25)
# usam UMA ÚNICA configuração fixa, para que a comparação entre os modelos
# (Casos 5-9) não seja confundida por diferenças de pré-processamento.
#
# Escolhemos "stopwords_stemming" (remoção de stopwords + stemming) como
# configuração padrão: ela reduz o vocabulário de 6813 para 4117 termos
# (conflando variações morfológicas como "ablate/ablating/ablation" em um
# único stem) e remove termos de função (stopwords) que não carregam
# conteúdo semântico e apenas adicionam ruído tanto ao TF-IDF quanto ao
# BM25. Esta é a configuração padrão mais comum na literatura clássica de
# RI para coleções pequenas e controladas como a Cranfield.
CONFIG = "stopwords_stemming"

doc_tokens = preprocessed[CONFIG]["docs"]
query_tokens_list = preprocessed[CONFIG]["queries"]

print(f"Configuração usada: {CONFIG}")
print(f"Documentos: {len(doc_tokens)} | Consultas: {len(query_tokens_list)}")


Configuração usada: stopwords_stemming
Documentos: 1400 | Consultas: 225


## Construção do modelo

In [3]:
# Construção do Modelo Vetorial
#
# Esquema de ponderação "ltc" (tf logarítmico x idf, normalizado em L2):
#   tf(t,d)  = 1 + log10(count(t,d))         se count(t,d) > 0, senão 0
#   idf(t)   = log10(N / df(t))
#   peso(t,d)= tf(t,d) * idf(t)
#   vetor(d) = peso(*,d) / ||peso(*,d)||_2
#
# A similaridade usada é o cosseno entre o vetor da consulta e o vetor de
# cada documento; como ambos os vetores estão normalizados em L2, o
# cosseno é o produto escalar entre eles. Toda a lógica de tf, idf e
# similaridade está implementada manualmente em src/vector_model.py
# (usamos scipy.sparse apenas como estrutura de dados eficiente para
# vetores esparsos, não como um "modelo pronto").
vsm = VectorSpaceModel(doc_tokens)
print(f"Vocabulário do modelo vetorial: {len(vsm.vocab)} termos")
print(f"Matriz documento-termo: {vsm.doc_matrix.shape}, "
      f"{vsm.doc_matrix.nnz} entradas não-nulas "
      f"({100*vsm.doc_matrix.nnz/(vsm.doc_matrix.shape[0]*vsm.doc_matrix.shape[1]):.2f}% de densidade)")


Vocabulário do modelo vetorial: 4292 termos
Matriz documento-termo: (1400, 4292), 83889 entradas não-nulas (1.40% de densidade)


## Exemplo de recuperação

Abaixo, o Top-10 de documentos recuperados pelo Modelo Vetorial para a
primeira consulta da coleção, indicando o grau de relevância (segundo o
qrels) de cada documento retornado.


In [4]:
# Exemplo: ranking para a primeira consulta da coleção
query_id_0 = query_ids[0]
print(f"Consulta {query_id_0}: {df_queries.iloc[0]['text']}")
print(f"Tokens (pós pré-processamento): {query_tokens_list[0]}")
print()

ranking = vsm.rank(query_tokens_list[0], top_n=10)
print("Top-10 documentos recuperados pelo Modelo Vetorial:")
rows = []
grades = dict(zip(df_qrels[df_qrels.query_id == query_id_0].doc_id,
                   df_qrels[df_qrels.query_id == query_id_0].relevance))
for rank, (doc_idx, score) in enumerate(ranking, start=1):
    did = doc_ids[doc_idx]
    grade = grades.get(did, None)
    rows.append({
        "rank": rank, "doc_id": did, "score_cosseno": round(score, 4),
        "grau_relevancia": grade if grade is not None else "não julgado",
        "titulo": doc_title[did][:80],
    })
display(pd.DataFrame(rows))


Consulta 1: what similarity laws must be obeyed when constructing aeroelastic models
of heated high speed aircraft .
Tokens (pós pré-processamento): ['similar', 'law', 'must', 'obey', 'construct', 'aeroelast', 'model', 'heat', 'high', 'speed', 'aircraft']

Top-10 documentos recuperados pelo Modelo Vetorial:


,rank,doc_id,score_cosseno,grau_relevancia,titulo
0,1,573,0.2346,não julgado,viscous hypersonic similitude .
1,2,944,0.2065,não julgado,one dimensional heat conduction through the sk...
2,3,51,0.1988,3,theory of aircraft structural models subjected...
3,4,184,0.1768,2,scale models for thermo-aeroelastic research .
4,5,878,0.1630,não julgado,experimental model techniques and equipment fo...
5,6,486,0.1620,-1,similarity laws for aerothermoelastic testing .
6,7,12,0.1558,3,some structural and aerelastic considerations ...
7,8,665,0.1513,não julgado,on the theory of hypersonic gas flow with a po...
8,9,879,0.1443,3,flutter model testing at transonic speeds .
9,10,1361,0.1281,não julgado,large deflections of structures subjected to h...


## Verificação de sanidade do IDF

Termos raros (que aparecem em poucos documentos) devem ter idf alto e
pesar mais nas comparações; termos muito comuns devem ter idf baixo. Isso
confirma que a ponderação está calculada corretamente antes de seguirmos
para o BM25 e a avaliação quantitativa.


In [5]:
# Verificação de sanidade: a consulta é mais similar a si mesma (como
# "pseudo-documento") do que documentos aleatórios são entre si, e termos
# muito frequentes na coleção (baixo idf) pesam pouco no ranking.
top_idf_terms = sorted(zip(vsm.vocab, vsm.idf), key=lambda x: -x[1])[:5]
bottom_idf_terms = sorted(zip(vsm.vocab, vsm.idf), key=lambda x: x[1])[:5]
print("Termos mais raros (maior idf) — carregam mais peso no cosseno:")
for t, v in top_idf_terms:
    print(f"  {t}: idf={v:.3f}")
print("\nTermos mais comuns (menor idf) — carregam menos peso:")
for t, v in bottom_idf_terms:
    print(f"  {t}: idf={v:.3f}")


Termos mais raros (maior idf) — carregam mais peso no cosseno:
  ab: idf=3.146
  abbrevi: idf=3.146
  absent: idf=3.146
  abundantli: idf=3.146
  academ: idf=3.146

Termos mais comuns (menor idf) — carregam menos peso:
  flow: idf=0.283
  result: idf=0.307
  number: idf=0.389
  pressur: idf=0.404
  effect: idf=0.414
